# 🛢️ PumpGuard AI — Workshop IoT Predictive Maintenance
**v2.0** &nbsp;·&nbsp; Duration: 3–4 hours &nbsp;·&nbsp; Level: Intermediate &nbsp;·&nbsp; Platform: Google Colab

---

## 🎯 Objectives

Build a real-world IoT Predictive Maintenance system step by step, covering:

| Layer | Technology | Role |
|-------|------------|------|
| Sensor | Python simulator | Simulates 52 industrial sensors |
| Transport | Mosquitto MQTT | Local broker, latency ~1ms |
| Pipeline | Node-RED | Processing & anomaly detection |
| Backend | FastAPI + WebSocket | API server, real-time broadcast |
| AI | Gemini / Claude / GPT-4 | Analysis & maintenance recommendations |
| Alerting | Resend | Automated email alerts |
| Dashboard | HTML / JS / Chart.js | Real-time browser interface |

---

## 🏗️ Architecture

```
[mqtt_replay.py] → [Mosquitto :1883] → [Node-RED :1880]
                                               │ HTTP POST
                                       [FastAPI :8000 on Colab]
                                               │ Cloudflare Tunnel
                                         [Dashboard – Browser]
```

---

## 📋 How to use this notebook

> Each part = **theory → practice → confirmation**.
> Source files are provided by the instructor — you will **upload each file** to Colab as instructed.


---
# ⚙️ Pre-Workshop Setup (Prerequisites)

Complete the following before starting the workshop:

---

## 1 · Google Colab

Sign in to **[colab.research.google.com](https://colab.research.google.com/)** with your Google account.
Select **Runtime → Change runtime type → T4 GPU** (CPU is also fine).

---

## 2 · Resend API key *(for email alerts)*

1. Go to **[resend.com](https://resend.com/)** → Sign up for free
2. Go to **API Keys** → create a new key → copy it
3. Save the following:
```
RESEND_KEY = re_xxxxxxxxxxxx
ALERT_EMAIL = your@email.com
```

---

## 3 · Gemini API key *(for AI analysis)*

1. Go to **[aistudio.google.com/apikey](https://aistudio.google.com/apikey)** → create a key
2. Save the following:
```
GEMINI_API_KEY = AIzaxxxxxxxxxxxxxxxx
```

---

## 4 · Prepare files (download from GitHub repo)

| File | Description |
|------|-------------|
| `server.py` | FastAPI backend |
| `index.html` | Main dashboard |
| `control.html` | Control panel |
| `flows.json` | Node-RED pipeline config |
| `sensor.csv` | Pump Sensor dataset (Kaggle) |
| `analyze_sensors.py` | Analyse CSV → generate config JSON |
| `mqtt_replay.py` | Replay sensor data over MQTT |


---
# Part 1 — Setup & Workspace Creation

## 1.1 Check Runtime


In [ ]:
import sys, platform, subprocess
print(f'Python  : {sys.version.split()[0]}')
print(f'Platform: {platform.platform()}')
mem = subprocess.run(['free','-h'], capture_output=True, text=True).stdout
print('Memory:\n' + mem)
assert sys.version_info >= (3,8), 'Python 3.8+ required'
print('✅ Runtime OK')


## 1.2 Create Project Directory Structure

Run this cell to create the folder structure.
You will upload files here as directed in each part of the notebook.

```
/content/pump-iot-demo/
├── backend/        ← server.py, .env          (Part 3)
├── dashboard/      ← index.html, control.html (Part 6)
├── nodered/        ← flows.json               (Part 5)
└── scripts/        ← mqtt_replay.py           (Part 7)
```


In [ ]:
import os
PROJ = '/content/pump-iot-demo'
for d in ['backend', 'dashboard', 'nodered', 'scripts']:
    os.makedirs(os.path.join(PROJ, d), exist_ok=True)
    print(f'  📁 {PROJ}/{d}/')
print('\n✅ Directory structure created. Ready to receive files!')


---
# Part 2 — Install Dependencies

## 2.1 Python packages

| Package | Role |
|---------|------|
| `fastapi` + `uvicorn` | Web framework + ASGI server |
| `paho-mqtt` | MQTT client for subscribing from the broker |
| `python-dotenv` | Read variables from `.env` file |
| `anthropic` / `openai` / `google-generativeai` | AI SDKs |
| `resend` | Email API |
| `pycloudflared` | Create public URL for Colab |


In [ ]:
print('📥 Installing Python packages...')
!pip install -q fastapi 'uvicorn[standard]' paho-mqtt python-dotenv 2>&1 | tail -3
print('✅ Done!')


## 2.2 Install Mosquitto & Node-RED

This workshop uses **Mosquitto** (local MQTT broker) and **Node-RED** (running on Colab, exposed via tunnel).
Run this cell once before getting started:


In [ ]:
print('📥 Installing Mosquitto...')
!apt-get install -y -q mosquitto mosquitto-clients 2>&1 | tail -3
print('📥 Installing Node.js 20 + Node-RED...')
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - 2>&1 | tail -3
!apt-get install -y -q nodejs 2>&1 | tail -3
!npm install -g --quiet node-red 2>&1 | tail -3
print('✅ Mosquitto + Node-RED installed successfully')


---
# Part 3 — Backend: `server.py`

## What does server.py do?

This file is the processing hub of the system:

```
           ┌─────────────────────────────────────┐
           │            server.py                │
           │                                     │
MQTT ──►  MQTTBridge ──► WebSocket ──► Dashboard │
           │                                     │
REST ──►  /analyze ──► AI API ──► JSON response  │
           │                                     │
REST ──►  /alert ──► Email (Resend)              │
           └─────────────────────────────────────┘
```

**Key endpoints:**

| Endpoint | Method | Purpose |
|----------|--------|---------|
| `/ws` | WebSocket | Real-time sensor stream to Dashboard |
| `/analyze` | POST | Trigger AI analysis |
| `/alert` | POST | Send email alert (called by Node-RED) |
| `/health` | GET | Status check |
| `/dashboard/` | GET | Serve dashboard UI |


### 📋 Practice: Upload `server.py`

*Main backend file — required before starting FastAPI.*

1. Open the **Files panel** on the left *(📁 icon or `Ctrl+Shift+F`)*
2. Navigate to: **`/content/pump-iot-demo/backend/`**
3. Drag and drop **`server.py`** into the folder

| &nbsp; | Path |
|--------|------|
| 📦 In ZIP | `pump-iot-demo/backend/server.py` |
| 📍 Upload to | `/content/pump-iot-demo/backend/server.py` |

> After uploading, run the cell below to verify.


In [ ]:
import os
_p = '/content/pump-iot-demo/backend/server.py'
assert os.path.exists(_p), '❌ server.py not found — please upload it following the instructions above'
_kb = max(1, os.path.getsize(_p) // 1024)
print(f'✅ server.py ({_kb} KB) — OK')


## 3.2 Create the `.env` file

The `.env` file stores API keys — **created via code** since it contains personal credentials. **Never** commit this file to Git.

**Required variables:**

| Variable | Description |
|----------|-------------|
| `MQTT_HOST` | Broker host — `localhost` (Mosquitto runs on Colab) |
| `MQTT_PORT` | `1883` (no TLS for local broker) |
| `AI_PROVIDER` | `gemini` / `claude` / `openai` |
| `GEMINI_API_KEY` (or equivalent) | AI API key |
| `RESEND_API_KEY` | (Optional) Email alerts via resend.com |
| `ALERT_FROM` / `ALERT_TO` | Sender / recipient email addresses |


In [ ]:
import os
# ════════════════════════════════════════
# ✏️ Fill in your details below
# ════════════════════════════════════════
# MQTT: using local Mosquitto — no changes needed
MQTT_HOST = 'localhost'
MQTT_PORT = 1883        # local, no TLS

AI_PROVIDER   = 'anthropic'   # 'anthropic' | 'openai' | 'gemini'
ANTHROPIC_KEY = ''
OPENAI_KEY    = ''
GEMINI_KEY    = ''

RESEND_KEY  = ''
ALERT_FROM  = 'alerts@yourdomain.com'
ALERT_TO    = 'your@email.com'
# ════════════════════════════════════════

env_lines = [
    '# PumpGuard AI — Environment Config',
    f'MQTT_HOST={MQTT_HOST}',
    f'MQTT_PORT={MQTT_PORT}',
    'MQTT_USERNAME=',
    'MQTT_PASSWORD=',
    'MQTT_TLS=false',
    f'AI_PROVIDER={AI_PROVIDER}',
    f'ANTHROPIC_API_KEY={ANTHROPIC_KEY}',
    f'OPENAI_API_KEY={OPENAI_KEY}',
    f'GEMINI_API_KEY={GEMINI_KEY}',
    f'RESEND_API_KEY={RESEND_KEY}',
    f'ALERT_FROM={ALERT_FROM}',
    f'ALERT_TO={ALERT_TO}',
]
env_path = '/content/pump-iot-demo/backend/.env'
with open(env_path, 'w') as f:
    f.write('\n'.join(env_lines))
print('✅ .env created: ' + env_path)
print()
print('Contents (values masked):')
with open(env_path) as f:
    for line in f:
        if '=' in line and not line.startswith('#'):
            k, v = line.strip().split('=', 1)
            masked = (v[:3]+'***') if len(v) > 4 else ('(empty)' if not v else v)
            print(f'  {k:25s} = {masked}')


---
# Part 4 — MQTT Broker

## What is MQTT?

**MQTT** is a lightweight pub/sub protocol — the industry standard for IoT.

```
[mqtt_replay.py]  ──publish──►  [MQTT Broker]  ──subscribe──►  [FastAPI → WebSocket → Dashboard]
                                  topic: pump/sensors
```

Workshop uses **Mosquitto** running directly on Colab — no cloud broker needed, **lower latency** (~1ms vs ~50ms for cloud MQTT).

---
## ☁️ Start Mosquitto & verify connection


---
## 4.1 Start Mosquitto (local MQTT broker)


In [ ]:
import subprocess, time, os

# Stop any existing mosquitto process
subprocess.run(['pkill', '-f', 'mosquitto'], capture_output=True)
time.sleep(0.5)

# Minimal config at /tmp — NO pid_file entry.
# When run with -c /tmp/mosquitto.conf, mosquitto only reads this file,
# ignoring /etc/mosquitto/mosquitto.conf (avoids pid dir permission errors).
with open('/tmp/mosquitto.conf', 'w') as f:
    f.write('listener 1883\nallow_anonymous true\n')

# Use Popen because mosquitto is a daemon — must keep running in the background
mosq_proc = subprocess.Popen(
    ['mosquitto', '-c', '/tmp/mosquitto.conf'],
    stdout=subprocess.DEVNULL, stderr=subprocess.PIPE
)
time.sleep(1)

if mosq_proc.poll() is None:
    print(f'✅ Mosquitto running (PID {mosq_proc.pid}) — port 1883')
    r = subprocess.run(
        ['mosquitto_pub', '-h', 'localhost', '-t', 'test', '-m', 'ok'],
        capture_output=True, timeout=3
    )
    print('   Publish test:', 'OK ✅' if r.returncode == 0 else r.stderr.decode())
else:
    err = mosq_proc.stderr.read().decode(errors='replace')
    print('❌ Mosquitto failed:\n' + err)


In [ ]:
# Verify paho-mqtt can connect to Mosquitto
import paho.mqtt.client as mqtt, os, threading
from dotenv import load_dotenv
load_dotenv('/content/pump-iot-demo/backend/.env')

HOST = os.environ.get('MQTT_HOST', 'localhost')
PORT = int(os.environ.get('MQTT_PORT', 1883))

_ok = threading.Event()
def _on_connect(c, ud, f, rc, p=None):
    if rc == 0:
        _ok.set()
        print('✅ paho-mqtt connected successfully → ' + HOST + ':' + str(PORT))
    else:
        print('❌ Connection failed, error code=' + str(rc))

c = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id='workshop-test')
c.on_connect = _on_connect
try:
    c.connect(HOST, PORT, keepalive=5)
    c.loop_start()
    connected = _ok.wait(timeout=5)
    c.disconnect(); c.loop_stop()
    if not connected:
        print('⚠️ Timeout — re-run the "Start Mosquitto" cell above')
except Exception as e:
    print('❌ ' + str(e))
    print('  → Re-run the "Start Mosquitto" cell above first')


---
# Part 5 — Node-RED Pipeline

## Sensor data processing pipeline

```
[MQTT In: pump/sensors]
       │
[Parse & Validate]      ← validate schema, drop NaN
       │
[Rolling Buffer 60]     ← keep last 60 readings (~30s) in RAM
       │
[Compute Trends]        ← avg, slope, std_dev, anomaly_score
       │
       ├─(score < 0.4)──► [POST /sensor-update]  ← update dashboard
       └─(score ≥ 0.4)──► [Throttle 60s] ──► [POST /alert]  ← AI + email
```

Node-RED runs **locally** on Colab and automatically uses the `flows.json` you upload.
After the public URL is created (Part 6), students can view the pipeline via **NODERED_URL**.


### 📋 Practice: Upload `flows.json`

*Node-RED pipeline config — must be uploaded before starting Node-RED.*

1. Open the **Files panel** on the left *(📁 icon or `Ctrl+Shift+F`)*
2. Navigate to: **`/content/pump-iot-demo/nodered/`**
3. Drag and drop **`flows.json`** into the folder

| &nbsp; | Path |
|--------|------|
| 📦 In ZIP | `pump-iot-demo/nodered/flows.json` |
| 📍 Upload to | `/content/pump-iot-demo/nodered/flows.json` |

> After uploading, run the cell below to verify.


In [ ]:
import os
_p = '/content/pump-iot-demo/nodered/flows.json'
assert os.path.exists(_p), '❌ flows.json not found — please upload it following the instructions above'
_kb = max(1, os.path.getsize(_p) // 1024)
print(f'✅ flows.json ({_kb} KB) — OK')


> 💡 **flows.json** is pre-configured to work with local Mosquitto (`localhost:1883`)
> and FastAPI (`localhost:8000`). Node-RED will be started in **Part 5.2** below.


---
## 5.2 Start Node-RED & Deploy Flow

Node-RED runs on Colab. The flow is deployed **automatically via REST API** — no need to open a browser or click Deploy.

> 💡 **Why not use the UI?**
> When accessing Node-RED through a Cloudflare tunnel, the editor WebSocket times out → the Deploy button is greyed out.
> Using the API (`POST /flows`) is the standard way to deploy flows from a script — faster and no manual steps needed.


In [ ]:
import subprocess, time, os, requests, json as _json

NR_HOME = '/root/.node-red'
os.makedirs(NR_HOME, exist_ok=True)

flow_src = '/content/pump-iot-demo/nodered/flows.json'
if not os.path.exists(flow_src):
    print('❌ flows.json not found — upload it in step 5.1 first')
else:
    with open(flow_src, 'rb') as r, open(NR_HOME + '/flows.json', 'wb') as w:
        w.write(r.read())

    with open(NR_HOME + '/settings.js', 'w') as f:
        f.write('module.exports = {\n'
                '  uiPort: 1880,\n'
                '  httpAdminRoot: "/",\n'
                '  userDir: "/root/.node-red",\n'
                '  flowFile: "flows.json",\n'
                '  logging: { console: { level: "warn" } }\n'
                '};\n')

    # Capture stderr to display errors if Node-RED crashes
    nr_proc = subprocess.Popen(
        ['node-red', '--settings', NR_HOME + '/settings.js'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        cwd=NR_HOME
    )

    print('⏳ Waiting for Node-RED to start...')
    ready = False
    for i in range(45):
        time.sleep(2)
        if nr_proc.poll() is not None:
            out = nr_proc.stdout.read().decode(errors='replace')
            print('❌ Node-RED crashed. Output:')
            print(out[-2000:] if len(out) > 2000 else out)
            break
        try:
            if requests.get('http://localhost:1880', timeout=2).status_code < 500:
                ready = True
                break
        except: pass
        print(f'   {(i+1)*2}s...', end='\r')

    if ready:
        print(f'✅ Node-RED running — port 1880 ({(i+1)*2}s)')
        try:
            with open(flow_src) as f:
                flows = _json.load(f)
            payload = flows if isinstance(flows, list) else flows.get('flows', [])
            r = requests.post('http://localhost:1880/flows', json=payload,
                headers={'Content-Type': 'application/json'},
                timeout=10)
            if r.status_code in (200, 204):
                print('✅ Flow deployed (REST API)')
            else:
                print(f'⚠️ Deploy HTTP {r.status_code}: {r.text[:300]}')
        except Exception as e:
            print(f'⚠️ Deploy error: {e}')
    elif nr_proc.poll() is None:
        print('⚠️ Timed out (90s) — try re-running this cell')


---
# Part 6 — Dashboard & FastAPI Server

## Dashboard consists of 2 files

| File | Description |
|------|-------------|
| `index.html` | Main dashboard: 4 tabs (Live Feed · Timeline · AI Recommendation · Sensor Status) |
| `control.html` | Control Panel: simulate Normal / Warning / Critical scenarios |

The dashboard connects to the backend via **WebSocket** for real-time data:
```javascript
const ws = new WebSocket(`wss://${location.host}/ws`);
ws.onmessage = e => {
    const data = JSON.parse(e.data);
    updateCharts(data);     // Chart.js timeline
    updateGauges(data);     // SVG arc gauges
    updateSensorPage(data); // Sensor status grid
}
```


### 📋 Practice: Upload `index.html`

*Main dashboard with 4 tabs, real-time charts and SVG gauges.*

1. Open the **Files panel** on the left *(📁 icon or `Ctrl+Shift+F`)*
2. Navigate to: **`/content/pump-iot-demo/dashboard/`**
3. Drag and drop **`index.html`** into the folder

| &nbsp; | Path |
|--------|------|
| 📦 In ZIP | `pump-iot-demo/dashboard/index.html` |
| 📍 Upload to | `/content/pump-iot-demo/dashboard/index.html` |

> After uploading, run the cell below to verify.


In [ ]:
import os
_p = '/content/pump-iot-demo/dashboard/index.html'
assert os.path.exists(_p), '❌ index.html not found — please upload it following the instructions above'
_kb = max(1, os.path.getsize(_p) // 1024)
print(f'✅ index.html ({_kb} KB) — OK')


### 📋 Practice: Upload `control.html`

*Control panel — used to simulate fault scenarios during the demo.*

1. Open the **Files panel** on the left *(📁 icon or `Ctrl+Shift+F`)*
2. Navigate to: **`/content/pump-iot-demo/dashboard/`**
3. Drag and drop **`control.html`** into the folder

| &nbsp; | Path |
|--------|------|
| 📦 In ZIP | `pump-iot-demo/dashboard/control.html` |
| 📍 Upload to | `/content/pump-iot-demo/dashboard/control.html` |

> After uploading, run the cell below to verify.


In [ ]:
import os
_p = '/content/pump-iot-demo/dashboard/control.html'
assert os.path.exists(_p), '❌ control.html not found — please upload it following the instructions above'
_kb = max(1, os.path.getsize(_p) // 1024)
print(f'✅ control.html ({_kb} KB) — OK')


## 6.1 Start FastAPI server

After uploading `server.py` (Part 3) and the 2 dashboard files, start the backend:


In [ ]:
import subprocess, time
fastapi_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'server:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='/content/pump-iot-demo/backend',
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
print('⏳ Waiting for FastAPI...')
for _ in range(20):
    time.sleep(1)
    line = fastapi_proc.stdout.readline().decode('utf-8', errors='replace')
    if 'Application startup complete' in line or 'Uvicorn running' in line:
        print(f'✅ FastAPI running (PID {fastapi_proc.pid}) | localhost:8000')
        break
else:
    if fastapi_proc.poll() is None:
        print('⚠️ Still starting... Run the health check cell below')
    else:
        print('❌ Exited early — check log:')
        print(fastapi_proc.stdout.read(2000).decode(errors='replace'))


In [ ]:
import requests, time
time.sleep(2)
try:
    r = requests.get('http://localhost:8000/health', timeout=5)
    print('✅ Health check: ' + str(r.status_code))
    print('   ' + str(r.json()))
except Exception as e:
    print('❌ ' + str(e))

## 6.2 Create Public URL

Colab does not have a public IP. We use a tunnel to expose FastAPI and Node-RED externally.

### Option A — ngrok *(recommended, stable WebSocket)*
1. Sign up for free at **[ngrok.com](https://ngrok.com/)** → copy your authtoken
2. Fill in `NGROK_TOKEN` in the cell below → students get full Node-RED editor access

### Option B — Cloudflare trycloudflare *(no account needed)*
- No sign-up required, but WebSocket may disconnect → Node-RED editor shows **connecting**
- Dashboard works fine

> ⏱️ URL is valid for ~2 hours. If it expires, re-run this cell.


In [ ]:
import os, re, subprocess, time, threading, requests

# ════════════════════════════════════════════════
# ✏️ Option A: Enter your ngrok token (leave empty → use Cloudflare)
NGROK_TOKEN = ''   # get yours at: https://dashboard.ngrok.com/get-started/your-authtoken
# ════════════════════════════════════════════════

PUBLIC_URL  = None
NODERED_URL = None

def _update_env_and_restart(api_url):
    global PUBLIC_URL, fastapi_proc
    PUBLIC_URL = api_url
    env_path = '/content/pump-iot-demo/backend/.env'
    if os.path.exists(env_path):
        with open(env_path) as f: content = f.read()
        content = re.sub(r'PUBLIC_URL=.*', 'PUBLIC_URL=' + PUBLIC_URL, content) \
            if 'PUBLIC_URL=' in content else content + '\nPUBLIC_URL=' + PUBLIC_URL
        with open(env_path, 'w') as f: f.write(content)
    if 'fastapi_proc' in dir() and fastapi_proc.poll() is None:
        fastapi_proc.terminate(); time.sleep(1)
    fastapi_proc = subprocess.Popen(
        ['python', '-m', 'uvicorn', 'server:app', '--host', '0.0.0.0', '--port', '8000'],
        cwd='/content/pump-iot-demo/backend',
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        env={**os.environ, 'PUBLIC_URL': PUBLIC_URL},
    )
    time.sleep(3)

if NGROK_TOKEN.strip():
    # ── Option A: ngrok ──────────────────────────────────────────────────
    !pip install -q pyngrok
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_TOKEN
    ngrok.kill()  # stop any existing tunnel

    api_tunnel = ngrok.connect(8000, 'http')
    nr_tunnel  = ngrok.connect(1880, 'http')
    _update_env_and_restart(api_tunnel.public_url)
    NODERED_URL = nr_tunnel.public_url
    print('🔌 Tunnel: ngrok (full WebSocket support)')

else:
    # ── Option B: Cloudflare trycloudflare ───────────────────────────────
    !pip install -q pycloudflared
    from pycloudflared import try_cloudflare

    _api_url = _nr_url = None
    def _cf_api():
        global _api_url
        r = try_cloudflare(port=8000)
        _api_url = r.tunnel if hasattr(r, 'tunnel') else str(r)
    def _cf_nr():
        global _nr_url
        r = try_cloudflare(port=1880)
        _nr_url = r.tunnel if hasattr(r, 'tunnel') else str(r)

    t1 = threading.Thread(target=_cf_api, daemon=True)
    t2 = threading.Thread(target=_cf_nr,  daemon=True)
    t1.start(); t2.start()
    t1.join(timeout=25); t2.join(timeout=25)

    if not _api_url:
        print('⚠️ Cloudflare tunnel failed — retry or fill in NGROK_TOKEN'); raise SystemExit
    _update_env_and_restart(_api_url)
    NODERED_URL = _nr_url
    print('🔌 Tunnel: Cloudflare (Node-RED editor may show "connecting")')

print()
print('=' * 65)
print('  🌐 DASHBOARD   : ' + PUBLIC_URL + '/dashboard/')
print('  🎮 CONTROLS    : ' + PUBLIC_URL + '/control/')
print('  📖 API DOCS    : ' + PUBLIC_URL + '/docs')
if NODERED_URL:
    print('  ⚙️  NODE-RED    : ' + NODERED_URL + ('  ✅' if NGROK_TOKEN.strip() else '  ⚠️ (WebSocket may be unstable)'))
print('=' * 65)
print('✅ FastAPI restarted with new PUBLIC_URL')


---
# Part 7 — Sensor Simulator

## Data pipeline

```
sensor.csv ──► analyze_sensors.py ──► sensor_groups.json
                                            │
mqtt_replay.py ◄────────────────────────────┘
     │
     └──► publish pump/sensors ──► MQTT Broker ──► Node-RED ──► Dashboard
```

## Required files

| File | Location | Purpose |
|------|----------|---------|
| `sensor.csv` | `data/` | Raw dataset — 220K rows, 52 sensors |
| `analyze_sensors.py` | `scripts/` | Analyse CSV → generate `sensor_groups.json` |
| `sensor_groups.json` | `data/` | **Auto-generated** after running analyze |
| `mqtt_replay.py` | `scripts/` | Replay CSV over MQTT with time compression |

> ⚡ **Time compression**: 1 minute of real data = ~0.17s in demo (360× compression).
> Dataset of 220K rows ≈ 153 days — full replay completes in ~10 minutes.


### 📋 Practice: Upload `sensor.csv` & `analyze_sensors.py`

**File 1 — Raw dataset:**
1. Open the **Files panel** on the left
2. Navigate to: **`/content/pump-iot-demo/data/`**
3. Upload **`sensor.csv`**

**File 2 — Analysis script:**
1. Navigate to: **`/content/pump-iot-demo/scripts/`**
2. Upload **`analyze_sensors.py`**

| &nbsp; | Source (ZIP) | Upload to |
|--------|--------------|-----------|
| `sensor.csv` | `pump-iot-demo/data/sensor.csv` | `/content/pump-iot-demo/data/sensor.csv` |
| `analyze_sensors.py` | `pump-iot-demo/scripts/analyze_sensors.py` | `/content/pump-iot-demo/scripts/analyze_sensors.py` |

> After uploading both files, run the cell below to generate `sensor_groups.json`.


In [ ]:
import os, subprocess, sys

# Check that both required files exist
csv_path    = '/content/pump-iot-demo/data/sensor.csv'
script_path = '/content/pump-iot-demo/scripts/analyze_sensors.py'

for _p, _name in [(csv_path, 'sensor.csv'), (script_path, 'analyze_sensors.py')]:
    assert os.path.exists(_p), '❌ ' + _name + ' not found — please upload it again'
    _kb = max(1, os.path.getsize(_p) // 1024)
    print('✅ ' + _name + ' (' + str(_kb) + ' KB) — OK')

# Run analyze_sensors.py to generate sensor_groups.json
print()
print('⚙️  Running analyze_sensors.py...')
result = subprocess.run(
    [sys.executable, 'scripts/analyze_sensors.py',
     '--csv', 'data/sensor.csv',
     '--out', 'data/sensor_groups.json'],
    cwd='/content/pump-iot-demo',
    capture_output=True, text=True
)
print(result.stdout[-1500:] if len(result.stdout) > 1500 else result.stdout)
if result.returncode != 0:
    print('❌ Error:', result.stderr[-500:])
else:
    grp_path = '/content/pump-iot-demo/data/sensor_groups.json'
    _kb = max(1, os.path.getsize(grp_path) // 1024)
    print('✅ sensor_groups.json (' + str(_kb) + ' KB) generated!')


### 📋 Practice: Upload `mqtt_replay.py`

*Sensor replay script — requires `sensor_groups.json` from the step above.*

1. Open the **Files panel** on the left
2. Navigate to: **`/content/pump-iot-demo/scripts/`**
3. Upload **`mqtt_replay.py`**

| &nbsp; | Path |
|--------|------|
| 📦 In ZIP | `pump-iot-demo/scripts/mqtt_replay.py` |
| 📍 Upload to | `/content/pump-iot-demo/scripts/mqtt_replay.py` |

## 7.1 Start the Simulator

After uploading, run the cell below. Open the **Dashboard URL** to see real-time data.

> 💡 The simulator starts from **row 0 (NORMAL state)** and replays the full dataset.
> Use **Part 10** to jump directly to the fault scenario during the demo.


In [ ]:
import subprocess, time, os, sys
from dotenv import load_dotenv
load_dotenv('/content/pump-iot-demo/backend/.env')

replay_path = '/content/pump-iot-demo/scripts/mqtt_replay.py'
if not os.path.exists(replay_path):
    print('❌ mqtt_replay.py not found — please upload it following the instructions above')
elif not os.path.exists('/content/pump-iot-demo/data/sensor_groups.json'):
    print('❌ sensor_groups.json not found — run the analyze cell in Part 7 first')
else:
    sim_proc = subprocess.Popen(
        [sys.executable, 'scripts/mqtt_replay.py',
         '--csv',       'data/sensor.csv',
         '--config',    'data/sensor_groups.json',
         '--row-start', '0'],  # Start from row 0 — NORMAL state
        cwd='/content/pump-iot-demo',
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    print('🚀 Simulator running — NORMAL state (starting from row 0)')
    print('   Dashboard will receive data in ~5s')
    print()
    for _ in range(15):
        line = sim_proc.stdout.readline().decode('utf-8', errors='replace')
        if not line: break
        print(line.rstrip())
    print()
    print('💡 Open Dashboard: ' + (PUBLIC_URL + '/dashboard/' if 'PUBLIC_URL' in dir() and PUBLIC_URL else '(run cell 6.2 first)'))


---
# Part 8 — End-to-End Pipeline Check

Full pipeline is now running:
```
[mqtt_replay.py]
  └─ MQTT pub → [Mosquitto :1883] → [Node-RED :1880]
                                          │ POST /sensor-update
                                  [FastAPI :8000]
                                          │ WebSocket
                                  [Dashboard Browser]
```

Run the cell below to check each layer:


In [ ]:
import requests, time
print('=== Pipeline Status ===')
time.sleep(1)
for name, url in [('FastAPI /health', 'http://localhost:8000/health'),
                  ('Latest sensor',   'http://localhost:8000/latest')]:
    try:
        r = requests.get(url, timeout=3)
        d = r.json()
        if name == 'FastAPI /health':
            mqtt_ok  = d.get('mqtt_connected', False)
            has_data = d.get('has_data', False)
            last_st  = d.get('last_status', 'N/A')
            broker   = d.get('mqtt_broker', 'N/A')
            ws       = d.get('ws_clients', 0)
            print('  ✅ FastAPI running')
            print('     MQTT broker   : ' + broker)
            print('     MQTT connected: ' + ('✅ yes' if mqtt_ok else '❌ NO — check .env credentials'))
            print('     Has live data : ' + ('✅ yes (status=' + str(last_st) + ')' if has_data else '⚠️  not yet — start mqtt_replay.py'))
            print('     WS clients    : ' + str(ws))
        else:
            if d.get('status') == 'no_data':
                print('  ⚠️  /latest: No data yet — start the simulator (Part 7)')
            else:
                print('  ✅ /latest: status=' + str(d.get('machine_status')) +
                      ' health=' + str(round(d.get('health_score', 0), 1)))
    except Exception as e:
        print('  ❌ ' + name + ': ' + str(e))

if 'PUBLIC_URL' in dir() and PUBLIC_URL:
    print()
    print('🌐 ' + str(PUBLIC_URL) + '/dashboard/')


---
# Part 9 — AI Analysis & Email Alerts

## Analysis flow

```
Node-RED (anomaly_score ≥ 0.4)
    └─ POST /alert (throttle 60s)
           └─ AI API → JSON response → WebSocket → Dashboard
           └─ Resend → HTML Email → Inbox
```


In [ ]:
import requests, json
test = {
    'machine_id': 'PUMP-001', 'status': 'warning',
    'vibration':   {'avg': 5.2,  'trend': 0.8,  'anomaly_score': 0.72},
    'temperature': {'avg': 88.5, 'trend': 0.3,  'anomaly_score': 0.65},
    'pressure':    {'avg': 8.9,  'trend': -0.1, 'anomaly_score': 0.45},
    'flow_rate':   {'avg': 135.2,'trend': -2.1, 'anomaly_score': 0.38},
}
print('📤 Testing AI analysis...')
try:
    _base = str(PUBLIC_URL) if 'PUBLIC_URL' in dir() and PUBLIC_URL else 'http://localhost:8000'
    r = requests.post(_base + '/alert', json=test, timeout=30)
    result = r.json()
    if result.get('status') == 'skipped':
        print('  ℹ️ ' + result.get('reason','') + ' (mock data will still appear on the dashboard)')
    else:
        ai = result.get('ai_analysis', {})
        print('  ✅ Risk: ' + str(ai.get('risk_level')))
        for rec in ai.get('recommendations', [])[:2]:
            print('  → ' + str(rec))
except Exception as e:
    print('❌ ' + str(e))


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv('/content/pump-iot-demo/backend/.env')
key = os.environ.get('RESEND_API_KEY', '')
if key and len(key) > 10:
    print('✅ Resend configured')
    print('   From : ' + os.environ.get('ALERT_FROM',''))
    print('   To   : ' + os.environ.get('ALERT_TO',''))
    print('   Emails sent on WARNING/CRITICAL (60s cooldown)')
else:
    print('ℹ️  Resend not configured — emails will be skipped')
    print('   To enable: sign up at resend.com → set RESEND_API_KEY in .env')


---
# Part 10 — Demo: Simulate a Fault

| Scenario | Symptoms | Real-world cause |
|----------|----------|-----------------|
| 🔴 Bearing Wear | Vibration rises, temperature increases | Worn bearing surfaces |
| 🔴 Cavitation | Pressure fluctuates, flow drops | Low inlet pressure |
| 🔴 Overheating | Temperature spikes rapidly | Insufficient lubrication |


In [ ]:
import subprocess, time, sys, os

print('🔴 Switching to CRITICAL mode...')
if 'sim_proc' in dir() and sim_proc.poll() is None:
    sim_proc.terminate()
    time.sleep(0.5)

sim_proc = subprocess.Popen(
    [sys.executable, 'scripts/mqtt_replay.py',
     '--csv',              'data/sensor.csv',
     '--config',           'data/sensor_groups.json',
     '--start-at-anomaly',
     '--anomaly-offset',   '0',
     '--quiet'],
    cwd='/content/pump-iot-demo',
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
print('✅ Simulator restarted — jumping directly to the BROKEN row')
print('   CRITICAL data will appear on the Dashboard in ~3s')
if 'PUBLIC_URL' in dir() and PUBLIC_URL:
    print('   🌐 ' + str(PUBLIC_URL) + '/dashboard/')


In [ ]:
import subprocess, sys, os

if 'sim_proc' in dir() and sim_proc.poll() is None:
    sim_proc.terminate()

sim_proc = subprocess.Popen(
    [sys.executable, 'scripts/mqtt_replay.py',
     '--csv',       'data/sensor.csv',
     '--config',    'data/sensor_groups.json',
     '--row-start', '0', '--quiet'],
    cwd='/content/pump-iot-demo',
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
print('✅ Reset to NORMAL mode — replaying from the start of the CSV')


---
# Part 11 — Monitoring & Troubleshooting


In [ ]:
import requests
def svc(name, proc, url=None):
    alive = proc is not None and proc.poll() is None
    icon = '🟢' if alive else '🔴'
    out = icon + ' ' + name
    if alive and url:
        try:
            r = requests.get(url, timeout=2)
            out += ' | HTTP ' + str(r.status_code)
        except: out += ' | unreachable'
    print(out)

print('═'*46)
svc('FastAPI :8000',   locals().get('fastapi_proc'), 'http://localhost:8000/health')
svc('Simulator',       locals().get('sim_proc'))
for n,v in [('Mosquitto','mosquitto_proc'),('Node-RED','nr_proc')]:
    if v in dir(): svc(n, eval(v))
print('═'*46)
if 'PUBLIC_URL' in dir() and PUBLIC_URL:
    print('\n🌐 ' + PUBLIC_URL + '/dashboard/')

| Symptom | Cause | Fix |
|---------|-------|-----|
| Dashboard blank | FastAPI not running | Re-run Part 6.1 |
| No data received | Node-RED not running | Re-run Part 5.2 |
| AI tab spinning | No API key configured | Add key → restart backend |
| Email not sent | RESEND_KEY invalid | Check resend.com Logs |
| MQTT error | Mosquitto not started | Re-run Part 4.1 |
| URL expired | Cloudflare ~2h limit | Re-run 6.2 → share new URL |
| Colab reset | Idle > 90 minutes | Re-run from Part 6 (files are preserved) |


---
# Part 12 — Wrap-up & Exercises

## 🎉 System complete!

| What you built | Production equivalent |
|---------------|----------------------|
| Mosquitto MQTT | AWS IoT Core / Azure IoT Hub |
| Node-RED | Industrial Edge Gateway |
| FastAPI + WebSocket | Real-time telemetry platform |
| AI analysis | Predictive maintenance ML |
| Resend email | PagerDuty / OpsGenie |

---

## 🚀 Extension exercises

**Beginner:**
1. Add an `acoustic_emission` sensor type (ultrasonic sound)
2. Adjust alert thresholds to match real pump datasheet values
3. Add a second language to the AI system prompt

**Intermediate:**
4. Save history to SQLite + add `/history?hours=24` endpoint
5. Add Basic Auth to the Control Panel
6. Integrate Grafana to visualise data from SQLite

**Advanced:**
7. Replace rule-based detection with **Isolation Forest** (scikit-learn)
8. Attach a real MPU-6050 sensor to a Raspberry Pi
9. Migrate to **AWS IoT Core + Lambda + DynamoDB**

---

**Docs:** [Mosquitto](https://mosquitto.org/documentation/) · [Node-RED](https://nodered.org/docs/) · [FastAPI](https://fastapi.tiangolo.com/)
